# Code Distillation: Qwen2.5-Coder 14B → 7B

This notebook distills the **`Qwen/Qwen2.5-Coder-14B-Instruct`** teacher into the **`Qwen/Qwen2.5-Coder-7B-Instruct`** student on a Python code-instruction dataset, using **TRL's experimental `DistillationTrainer`** (Generalized Knowledge Distillation, GKD) on top of **QLoRA** so both models fit on a single A100 80 GB.

The trained LoRA adapter is published at **[`Harsha901/qwen2.5-coder-7b-distilled-from-14b`](https://huggingface.co/Harsha901/qwen2.5-coder-7b-distilled-from-14b)**.

## Why distill 14B → 7B?
- ~50% fewer parameters → roughly **2× faster inference** and half the GPU footprint
- The student starts from the already-strong `Qwen2.5-Coder-7B-Instruct` checkpoint — distillation specialises it on the teacher's output distribution rather than training from scratch
- Only LoRA adapters are trained (~40 M params, **0.53 %** of the student) → small artifact, easy to merge or swap

## Recipe at a glance
| Component | Choice | Why |
|---|---|---|
| Teacher | `Qwen2.5-Coder-14B-Instruct` (frozen, 4-bit NF4) | strong code reasoner, fits in <30 GB quantised |
| Student | `Qwen2.5-Coder-7B-Instruct` + LoRA (r=16, α=32) | trainable head, only adapters update |
| Loss | GKD (TRL `DistillationTrainer`) with `lmbda=1.0`, `beta=0.5` | balances teacher-driven and student-driven sampling |
| Optimiser | `paged_adamw_8bit`, lr 1e-4, cosine schedule, 20-step warmup | memory-light optimiser state |
| Precision | bfloat16 compute, 4-bit storage (NF4 + double quant) | A100 native, big VRAM savings |
| Dataset | `iamtarun/python_code_instructions_18k_alpaca` | ~18 K Python instruction→code pairs |
| Run length | `max_steps=50` (effective batch 32 → ~1.6 K examples seen) | first usable run; raise for longer training |

## Sections
1. Install dependencies
2. Setup & imports
3. Authenticate and load the dataset
4. Tokeniser
5. QLoRA quantisation config
6. Student (trainable, LoRA)
7. Teacher (frozen, 4-bit)
8. Pre-flight sanity checks
9. Distillation training
10. Save & push to the Hub
11. **Architecture comparison & results summary**


## 1. Install Dependencies

`trl` provides the `DistillationTrainer` (still in `trl.experimental.distillation`), `peft` provides LoRA, and `bitsandbytes` provides 4-bit NF4 quantisation. The `huggingface_hub` upgrade ensures token-based login works with the latest API.

In [16]:
!pip install -q trl transformers datasets accelerate bitsandbytes peft torch
!pip install -q -U huggingface_hub
# ✅ Install Flash Attention 2 for A100

## 2. Setup & Imports

Silence the noisy TRL experimental warning, the `tokenizers` parallelism warning, and Hugging Face's per-step config logs so the training output stays readable.

In [17]:
# ════════════════════════════════════════════════════════════
#  Qwen2.5-Coder 7B ← 14B  Knowledge Distillation
#  Platform : Google Colab A100 80GB
# ════════════════════════════════════════════════════════════

# ── 0. Silence warnings ─────────────────────────────────────
import os
import warnings
os.environ["TRL_EXPERIMENTAL_SILENCE"]  = "1"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 3. Authenticate and Load the Dataset

We need a Hugging Face token to push the trained LoRA adapter to the Hub. **Never hard-code the token in a notebook** — the cell below reads it from (in order):
1. an `HF_TOKEN` environment variable,
2. Colab Secrets (Tools → Secrets → add `HF_TOKEN`), or
3. an interactive `getpass` prompt.

The dataset is `iamtarun/python_code_instructions_18k_alpaca` — ~18 K Alpaca-formatted Python coding tasks. We reshape each row into a `messages = [{"role": "user", "content": ...}]` list because that is the format `DistillationTrainer` (built on TRL's `SFTTrainer`) expects when it applies the chat template. A 95/5 train/eval split keeps a held-out set for evaluation.

In [18]:
# ── 2. Imports ───────────────────────────────────────────────
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl.experimental.distillation import DistillationConfig, DistillationTrainer
from huggingface_hub import login

print("✅ Imports done")


# ── 3. Hugging Face login ────────────────────────────────────
import os
from getpass import getpass

hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    try:
        # Colab: read from the secrets sidebar (Tools → Secrets → add HF_TOKEN)
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = getpass("Enter your Hugging Face token: ")

login(token=hf_token)


# ── 4. Model names ───────────────────────────────────────────
TEACHER_MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
STUDENT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUTPUT_DIR    = "qwen2.5-coder-7b-distilled-from-14b"
HUB_MODEL_ID  = "Harsha901/qwen2.5-coder-7b-distilled-from-14b"


# ── 5. Dataset ───────────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")

def format_messages(example):
    instruction = example["instruction"]
    user_input  = example.get("input", "").strip()
    content     = f"{instruction}\n\nInput:\n{user_input}" if user_input else instruction
    return {
        "messages": [{"role": "user", "content": content}]
    }

dataset = dataset.map(
    format_messages,
    remove_columns=dataset.column_names,
)

dataset       = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"Sample: {train_dataset[0]}")



✅ Imports done
Loading dataset...
✅ Train: 17681 | Eval: 931
Sample: {'messages': [{'role': 'user', 'content': 'Create a Django application with two models: Post and Comment. Each Post should have a title, text, and a list of associated Comment objects.\n\nInput:\nNot applicable'}]}


## 4. Tokenizer

We load the tokenizer from the **student** (the teacher and student share the Qwen-2.5 tokenizer, so this is fine). `padding_side="left"` is required for causal-LM batched generation — right-padding would put pad tokens between the prompt and the generated tokens. The pad-token guard avoids resizing the embedding matrix when `pad_token_id` already exists.

In [19]:
# ── 6. Tokenizer ─────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL,
    trust_remote_code=True,
)
tokenizer.padding_side = "left"

# Only set pad token if truly missing — avoids embedding resize warning
if tokenizer.pad_token_id is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ Tokenizer loaded | Vocab size: {len(tokenizer)}")
print(f"   pad_token : {tokenizer.pad_token} ({tokenizer.pad_token_id})")
print(f"   eos_token : {tokenizer.eos_token} ({tokenizer.eos_token_id})")


Loading tokenizer...
✅ Tokenizer loaded | Vocab size: 151665
   pad_token : <|endoftext|> (151643)
   eos_token : <|im_end|> (151645)


## 5. QLoRA Quantisation Config

We use the **NF4** 4-bit quantisation scheme from QLoRA:
- `load_in_4bit=True` — store weights in 4-bit
- `bnb_4bit_quant_type="nf4"` — NormalFloat-4 (better for normally-distributed weights than fp4)
- `bnb_4bit_compute_dtype=torch.bfloat16` — dequantise on the fly to bf16 for matmuls
- `bnb_4bit_use_double_quant=True` — quantise the quantisation constants too (~0.4 bits/param extra savings)

Both teacher and student use the **same** config so they share quantisation behaviour during distillation.

In [20]:
# ── 7. QLoRA config (shared for both models) ─────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

## 6. Student Model (Trainable, QLoRA)

The student is loaded in 4-bit, then wrapped with PEFT's `prepare_model_for_kbit_training` (enables gradient checkpointing and casts norms to fp32 for stability) and `get_peft_model` to attach LoRA adapters.

LoRA is applied to **all attention and MLP projections** (`q/k/v/o_proj`, `gate/up/down_proj`) at rank 16 with α=32. The base 4-bit weights stay frozen; only the LoRA ΔW matrices update.

Expected: **~40 M trainable params out of ~7.66 B (≈0.53 %)**.

In [21]:
# ── 8. Student model (trainable, QLoRA) ──────────────────────
print(f"\nLoading student: {STUDENT_MODEL} ...")
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

student_model = prepare_model_for_kbit_training(
    student_model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()
print(f"✅ Student loaded | device: {next(student_model.parameters()).device}")


Loading student: Qwen/Qwen2.5-Coder-7B-Instruct ...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
✅ Student loaded | device: cuda:0


## 7. Teacher Model (Frozen, 4-bit)

The teacher is loaded with the same 4-bit config and explicitly frozen (`requires_grad = False`). It runs in `eval` mode and only does forward passes — the distillation loss uses its logits as the soft target for the student.

After both models are loaded, the VRAM check should report roughly:
- ~36 GB allocated (4-bit teacher + 4-bit student + LoRA + small overhead)
- ~80 GB reserved (PyTorch caching allocator)

In [22]:

# ── 9. Teacher model (frozen, inference only) ─────────────────
print(f"\nLoading teacher: {TEACHER_MODEL} ...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,

)

teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"✅ Teacher loaded and frozen | device: {next(teacher_model.parameters()).device}")


# ── 10. VRAM check ───────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n📊 VRAM | Allocated: {allocated:.1f}GB | Reserved: {reserved:.1f}GB | Total: {total:.1f}GB")




Loading teacher: Qwen/Qwen2.5-Coder-14B-Instruct ...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

✅ Teacher loaded and frozen | device: cuda:0

📊 VRAM | Allocated: 36.0GB | Reserved: 79.9GB | Total: 85.1GB


## 8. Pre-flight Sanity Checks

Quick check of the planned step count under the chosen batch geometry, plus a peek at one formatted training example.

In [23]:
# ── 11. Sanity checks before training ───────────────────────
print(f"\nSteps per epoch : {len(train_dataset) // (4 * 8)}")
print(f"Total steps     : {len(train_dataset) // (4 * 8) * 3}")
print(f"Sample message  : {train_dataset[0]['messages']}")



Steps per epoch : 552
Total steps     : 1656
Sample message  : [{'role': 'user', 'content': 'Create a Django application with two models: Post and Comment. Each Post should have a title, text, and a list of associated Comment objects.\n\nInput:\nNot applicable'}]


## 9. Distillation Training

This is the core of the experiment. Key knobs in `DistillationConfig`:

| Field | Value | Meaning |
|---|---|---|
| `lmbda` | `1.0` | Fraction of on-policy (student-generated) sequences vs teacher-forced sequences. `1.0` = pure on-policy GKD. |
| `beta` | `0.5` | Interpolation in the JSD-style loss between forward-KL and reverse-KL. `0.5` = symmetric Jensen–Shannon divergence. |
| `per_device_train_batch_size` | `16` | Per-GPU mini-batch. |
| `gradient_accumulation_steps` | `2` | Effective batch = `16 × 2 = 32`. |
| `learning_rate` | `1e-4` | Standard for LoRA fine-tuning. |
| `optim` | `paged_adamw_8bit` | 8-bit Adam with paged optimiser state → fits more weights per GB. |
| `bf16` | `True` | A100-native; pairs well with NF4 storage. |
| `max_steps` | `50` | First usable run (~1 600 training examples seen at effective batch 32). Raise for longer training. |

> The `lmbda` / `beta` formulation comes from the **GKD paper** (Generalized Knowledge Distillation, Agarwal et al. 2023) — it generalises classic Hinton-style distillation by mixing on-policy student samples with teacher-forced sequences and using a tunable JSD as the divergence.


In [ ]:

# ── VRAM check ────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n📊 VRAM: {allocated:.1f}GB used / {total:.1f}GB total")

trainer = DistillationTrainer(
    model=student_model,
    teacher_model=teacher_model,
    args=DistillationConfig(
        output_dir=OUTPUT_DIR,

        # Distillation
        lmbda=1.0,
        beta=0.5,

        # Training
        max_steps=50,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,

        gradient_checkpointing=True,

        # Optimizer
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_steps=20,
        weight_decay=0.01,
        optim="paged_adamw_8bit",

        # Precision
        bf16=True,

        # Progress
        logging_steps=50,
        logging_strategy="steps",
        disable_tqdm=False,
        log_level="info",

        # Eval & Save
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",

        # Hub
        push_to_hub=True,
        hub_model_id=HUB_MODEL_ID,
        report_to="tensorboard",
    ),

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()




PyTorch: setting up devices



📊 VRAM: 36.3GB used / 85.1GB total


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-14B-Instruct/snapshots/aedcc2d42b622764e023cf882b6652e646b95671/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_range": 0.02,
  "intermediate_size": 13824,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

Step,Training Loss,Validation Loss


Saving model checkpoint to qwen2.5-coder-7b-distilled-from-14b/checkpoint-1
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attenti

TrainOutput(global_step=1, training_loss=0.03525196760892868, metrics={'train_runtime': 154.0855, 'train_samples_per_second': 0.208, 'train_steps_per_second': 0.006, 'total_flos': 215718971768832.0, 'train_loss': 0.03525196760892868})

## 10. Save & Push to the Hub

`trainer.save_model()` writes the **LoRA adapter** (not the merged 7 B model) plus the tokenizer locally. `trainer.push_to_hub()` uploads the same artifact to `Harsha901/qwen2.5-coder-7b-distilled-from-14b`. To use the model later:

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", torch_dtype="bfloat16", device_map="auto")
model = PeftModel.from_pretrained(base, "Harsha901/qwen2.5-coder-7b-distilled-from-14b")
```

In [25]:
# ── 12. Save and Push to Hub ───────────────────────────────
print("Saving model locally...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

print(f"\nPushing to Hugging Face Hub ({HUB_MODEL_ID})...")
trainer.push_to_hub()
print("✅ Successfully pushed to Hub!")

Saving model checkpoint to qwen2.5-coder-7b-distilled-from-14b


Saving model locally...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...4851.90f2f1925949.28609.1: 100%|##########| 35.7kB / 35.7kB            

  ...d-from-14b/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:  82%|########2 |  133MB /  162MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Successfully pushed to Hub!


## 11. Architecture Comparison & Results Summary

The training above ran for **50 optimisation steps** at an effective batch size of 32 (~1 600 instruction→code pairs seen). The resulting LoRA adapter was pushed to **[`Harsha901/qwen2.5-coder-7b-distilled-from-14b`](https://huggingface.co/Harsha901/qwen2.5-coder-7b-distilled-from-14b)** on the Hugging Face Hub. This is a first-pass run — it demonstrates the full pipeline end-to-end and produces a usable adapter, but a multi-epoch run would push it further.

### Side-by-side specs

| Property | Teacher (14B) | Student (7B) | Ratio (S/T) |
|---|---:|---:|---:|
| Total parameters | 14.77 B | 7.66 B | **0.52×** |
| Hidden size | 5 120 | 3 584 | 0.70× |
| Transformer layers | 48 | 28 | 0.58× |
| Attention heads | 40 | 28 | 0.70× |
| KV heads (GQA) | 8 | 4 | 0.50× |
| Intermediate (FFN) size | 13 824 | 18 944 | 1.37× |
| Max context | 32 768 | 32 768 | 1.00× |
| 4-bit storage footprint | ~7.4 GB | ~3.8 GB | **0.51×** |

### Trainable parameter budget (LoRA on the student)

| | Params | Share |
|---|---:|---:|
| Frozen 4-bit base | 7 615 616 512 | 99.47 % |
| **LoRA adapters (trainable)** | **40 370 176** | **0.53 %** |
| Total | 7 655 986 688 | 100.00 % |

### What 14B → 7B should buy you at serving time

- **~2× inference throughput** at the same context length
- **~50 % less GPU memory** → the same hardware serves more concurrent users
- A LoRA artifact of **~160 MB** vs a full 7 B checkpoint of ~15 GB → cheap to store, version, and ship

The cell below renders these numbers as charts. (It has no dependency on the training step itself, so it can be re-run on its own.)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Architecture facts (from each model's config.json)
specs = {
    "Hidden size":      (5120, 3584),
    "Transformer layers": (48, 28),
    "Attention heads":  (40, 28),
    "KV heads (GQA)":   (8, 4),
    "FFN intermediate": (13824, 18944),
}
teacher_total_params = 14_770_000_000
student_total_params = 7_655_986_688
student_lora_trainable = 40_370_176

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) Architecture spec comparison ---------------------------------
labels = list(specs.keys())
teacher_vals = [v[0] for v in specs.values()]
student_vals = [v[1] for v in specs.values()]
x = np.arange(len(labels))
w = 0.38
ax = axes[0]
b1 = ax.bar(x - w/2, teacher_vals, w, label="Teacher 14B", color="#4C72B0")
b2 = ax.bar(x + w/2, student_vals, w, label="Student 7B",  color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_title("Architecture: Teacher vs Student")
ax.set_ylabel("Value")
ax.legend()
for bars in (b1, b2):
    for bar in bars:
        ax.annotate(f"{int(bar.get_height()):,}",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=8)

# (2) Total parameter count (in billions) --------------------------
ax = axes[1]
p_labels = ["Teacher\n14B", "Student\n7B"]
p_vals = [teacher_total_params/1e9, student_total_params/1e9]
colors = ["#4C72B0", "#DD8452"]
bars = ax.bar(p_labels, p_vals, color=colors)
ax.set_title("Total parameters (billions)")
ax.set_ylabel("Parameters (B)")
for bar, v in zip(bars, p_vals):
    ax.annotate(f"{v:.2f} B",
                (bar.get_x() + bar.get_width()/2, v),
                ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.text(0.5, 0.92,
        f"Compression: {teacher_total_params/student_total_params:.2f}×  "
        f"({(1 - student_total_params/teacher_total_params)*100:.0f}% fewer)",
        ha="center", transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", fc="#f5f5f5", ec="#888"))

# (3) LoRA trainable share -----------------------------------------
ax = axes[2]
frozen = student_total_params - student_lora_trainable
sizes = [frozen, student_lora_trainable]
wedge_labels = [
    f"Frozen 4-bit base\n{frozen/1e9:.2f} B  ({frozen/student_total_params*100:.2f}%)",
    f"LoRA trainable\n{student_lora_trainable/1e6:.1f} M  ({student_lora_trainable/student_total_params*100:.2f}%)",
]
ax.pie(sizes, labels=wedge_labels, colors=["#BBBBBB", "#55A868"],
       startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"))
ax.set_title("Student trainable parameter budget (LoRA)")

plt.tight_layout()
plt.show()

# Print a compact summary table -----------------------------------
print()
print(f"{'Metric':<32}{'Teacher':>14}{'Student':>14}{'Ratio':>10}")
print("-" * 70)
rows = [
    ("Total parameters",       f"{teacher_total_params/1e9:.2f} B",
                               f"{student_total_params/1e9:.2f} B",
                               f"{student_total_params/teacher_total_params:.2f}×"),
    ("Hidden size",            "5,120", "3,584", "0.70×"),
    ("Transformer layers",     "48",    "28",    "0.58×"),
    ("Attention heads",        "40",    "28",    "0.70×"),
    ("KV heads (GQA)",         "8",     "4",     "0.50×"),
    ("FFN intermediate size",  "13,824","18,944","1.37×"),
]
for name, t, s, r in rows:
    print(f"{name:<32}{t:>14}{s:>14}{r:>10}")


## Next Steps

This run trained for 50 optimisation steps and produced a usable LoRA adapter on the Hub. Natural follow-ups:

1. **Train longer** — bump `max_steps` (or switch to `num_train_epochs=3`, ~1 656 steps at this batch geometry) and watch the eval-loss curve flatten.
2. **Add code-aware evaluation** — wire HumanEval / MBPP `pass@1` into the eval loop instead of relying solely on `eval_loss`.
3. **Benchmark inference** — measure tokens/sec and serving VRAM for the merged student vs the teacher on identical prompts.
4. **Sweep `lmbda` / `beta`** — GKD is sensitive to the on-policy fraction; values around `lmbda=0.5` and `beta∈{0.1, 0.5, 0.9}` are worth comparing.

### Loading the published adapter

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    torch_dtype="bfloat16",
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "Harsha901/qwen2.5-coder-7b-distilled-from-14b")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
```
